In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec, FastText

# 1. Cargar datos
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

# IMPORTANTE: Pon aquí la columna que ganó tu torneo de preprocesamiento
X_texto = df['text']

# 2. Word2Vec necesita listas de palabras, no frases enteras. 
# Convertimos cada informe en una lista de tokens.
X_tokens = X_texto.apply(lambda x: str(x).split())

# 3. Split (Hacemos el split ANTES de entrenar Word2Vec para no hacer trampa)
X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    X_tokens, y, test_size=0.20, random_state=42, stratify=y
)

print("🧠 Entrenando modelo Word2Vec médico desde cero...")

# 4. Entrenar Word2Vec solo con el Train
# vector_size=100 (cada palabra será un vector de 100 dimensiones)
# window=5 (mira 5 palabras a la izquierda y 5 a la derecha para entender el contexto)
# min_count=2 (ignora palabras que salen menos de 2 veces)
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=2, workers=4)

# 5. Función para convertir un informe entero en un solo vector
# (Hacemos la media de los vectores de todas las palabras de ese informe)
def document_vector(doc_tokens, model):
    # Filtramos palabras que el modelo conoce
    palabras_conocidas = [word for word in doc_tokens if word in model.wv.key_to_index]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    # Calculamos la media de los vectores de esas palabras
    return np.mean(model.wv[palabras_conocidas], axis=0)

print("🔄 Convirtiendo textos a vectores promediados...")

# 6. Transformar Train y Test a vectores numéricos
X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_test_tokens])

# 7. Entrenar y evaluar con un modelo clásico (Regresión Logística para comparar)
print("🚀 Entrenando clasificador sobre Embeddings...")
modelo_lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
modelo_lr.fit(X_train_w2v, y_train)

score_w2v = f1_score(y_test, modelo_lr.predict(X_test_w2v), average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con Word2Vec (Propios) : {score_w2v:.4f}")
print("-" * 50)

🧠 Entrenando modelo Word2Vec médico desde cero...
🔄 Convirtiendo textos a vectores promediados...
🚀 Entrenando clasificador sobre Embeddings...
--------------------------------------------------
📊 Macro-F1 con Word2Vec (Propios) : 0.4496
--------------------------------------------------


In [2]:
print("🚀 Entrenando modelo FastText médico desde cero...")

# 4. Entrenar FastText solo con el Train
# vector_size=100 (cada palabra será un vector de 100 dimensiones)
# window=5 (mira 5 palabras a la izquierda y 5 a la derecha para entender el contexto)
# min_count=2 (ignora palabras que salen menos de 2 veces)
ft_model = FastText(sentences=X_train_tokens, vector_size=100, window=5, min_count=2, workers=4, seed=42)

# 5. Función para convertir un informe entero en un solo vector promediado
def document_vector_ft(doc_tokens, model):
    # A diferencia de Word2Vec, FastText puede inferir vectores de palabras que no ha visto (OOV)
    # gracias a que aprende n-gramas (trozos de letras).
    palabras_conocidas = [word for word in doc_tokens if word in model.wv]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    # Calculamos la media de los vectores
    return np.mean(model.wv[palabras_conocidas], axis=0)

print("🔄 Convirtiendo textos a vectores promediados con FastText...")

# 6. Transformar Train y Test a vectores numéricos
X_train_ft = np.array([document_vector_ft(tokens, ft_model) for tokens in X_train_tokens])
X_test_ft = np.array([document_vector_ft(tokens, ft_model) for tokens in X_test_tokens])

# 7. Entrenar y evaluar con Regresión Logística
print("⚖️ Entrenando clasificador sobre Embeddings FastText...")
modelo_lr_ft = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
modelo_lr_ft.fit(X_train_ft, y_train)

score_ft = f1_score(y_test, modelo_lr_ft.predict(X_test_ft), average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con FastText (Propios) : {score_ft:.4f}")
print("-" * 50)

🚀 Entrenando modelo FastText médico desde cero...
🔄 Convirtiendo textos a vectores promediados con FastText...
⚖️ Entrenando clasificador sobre Embeddings FastText...
--------------------------------------------------
📊 Macro-F1 con FastText (Propios) : 0.4568
--------------------------------------------------


In [5]:
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-100")

def document_vector_glove(doc_tokens, model):
    palabras_conocidas = [word for word in doc_tokens if word in model.key_to_index]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model[palabras_conocidas], axis=0)

print("🔄 Convirtiendo textos médicos a vectores GloVe...")
X_train_glove = np.array([document_vector_glove(tokens, glove_model) for tokens in X_train_tokens])
X_test_glove = np.array([document_vector_glove(tokens, glove_model) for tokens in X_test_tokens])

print("⚖️ Entrenando clasificador sobre Embeddings GloVe...")
modelo_lr_glove = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
modelo_lr_glove.fit(X_train_glove, y_train)

score_glove = f1_score(y_test, modelo_lr_glove.predict(X_test_glove), average='macro')
print(score_glove)
print("-" * 50)

🔄 Convirtiendo textos médicos a vectores GloVe...
⚖️ Entrenando clasificador sobre Embeddings GloVe...
0.42677639221876557
--------------------------------------------------


In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

print("📄 Preparando documentos etiquetados para Doc2Vec...")
# 4. Formatear datos de entrenamiento a TaggedDocument (Requisito de Doc2Vec)
train_tagged = [TaggedDocument(words=tokens, tags=[str(i)]) for i, tokens in enumerate(X_train_tokens)]

print("🧠 Entrenando modelo Doc2Vec...")
# 5. Entrenamos el modelo (epochs=20 suele ser recomendable para que entienda bien los documentos)
d2v_model = Doc2Vec(documents=train_tagged, vector_size=100, window=5, min_count=2, workers=4, epochs=20, seed=42)

print("🔄 Infiriendo vectores únicos para cada historial médico...")
# 6. Transformar Train y Test (Doc2Vec tiene su propia función para inferir el vector de un documento nuevo)
X_train_d2v = np.array([d2v_model.infer_vector(tokens) for tokens in X_train_tokens])
X_test_d2v = np.array([d2v_model.infer_vector(tokens) for tokens in X_test_tokens])

print("⚖️ Entrenando clasificador sobre Embeddings Doc2Vec...")
# 7. Entrenar y evaluar
modelo_lr_d2v = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
modelo_lr_d2v.fit(X_train_d2v, y_train)

score_d2v = f1_score(y_test, modelo_lr_d2v.predict(X_test_d2v), average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con Doc2Vec: {score_d2v:.4f}")
print("-" * 50)

📄 Preparando documentos etiquetados para Doc2Vec...
🧠 Entrenando modelo Doc2Vec...
🔄 Infiriendo vectores únicos para cada historial médico...
⚖️ Entrenando clasificador sobre Embeddings Doc2Vec...
--------------------------------------------------
📊 Macro-F1 con Doc2Vec (Gensim) : 0.4101
--------------------------------------------------
